In [11]:
"""
PCM Heart Sound Analysis Pipeline
GitHub Compatible - Relative Paths Only
Notebook Path: /model folder
"""
import numpy as np
import torch
import os
import shutil
import subprocess
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# -------------------------- Device Configuration --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------- Model Architecture --------------------------
class FCGRUCell(torch.nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.fc = torch.nn.Linear(input_size + hidden_size, 3 * hidden_size)
        self.hidden_size = hidden_size
        
    def forward(self, x, h_prev):
        combined = torch.cat([x, h_prev], dim=-1)
        reset_gate, update_gate, candidate = torch.split(self.fc(combined), self.hidden_size, dim=-1)
        reset_gate = torch.sigmoid(reset_gate)
        update_gate = torch.sigmoid(update_gate)
        candidate = torch.tanh(torch.cat([x, reset_gate * h_prev], dim=-1)[..., :self.hidden_size])
        return (1 - update_gate) * candidate + update_gate * h_prev

class SingleLayerGRU(torch.nn.Module):
    def __init__(self, input_size=16, hidden_size=16, batch_first=True):
        super().__init__()
        self.gru_cell = FCGRUCell(input_size, hidden_size)
        self.hidden_size = hidden_size
        self.batch_first = batch_first
    
    def forward(self, x):
        if self.batch_first:
            x = x.permute(1, 0, 2)
        h = torch.zeros(1, x.shape[1], self.hidden_size, device=x.device)[0]
        for t in range(x.shape[0]):
            h = self.gru_cell(x[t], h)
        return h.unsqueeze(0)

class HeartSoundModel(torch.nn.Module):
    def __init__(self, input_channels=2, num_classes=2):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv1d(input_channels,8,3,2,1), torch.nn.ReLU(),
            torch.nn.Conv1d(8,16,3,1,1), torch.nn.ReLU(), torch.nn.MaxPool1d(2),
            torch.nn.Conv1d(16,16,3,1,1), torch.nn.ReLU(), torch.nn.MaxPool1d(2),
            torch.nn.Conv1d(16,16,3,1,1), torch.nn.ReLU(), torch.nn.MaxPool1d(2),
            torch.nn.Conv1d(16,16,3,1,1), torch.nn.ReLU(), torch.nn.MaxPool1d(2)
        )
        self.gru = SingleLayerGRU(16,16)
        self.fc = torch.nn.Linear(16, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        return self.fc(self.gru(x.permute(0,2,1)).squeeze(0))

# -------------------------- Dataset Class --------------------------
class HeartSoundDataset(Dataset):
    def __init__(self, X):
        processed = []
        for seg in X:
            seg = seg[:4096] if len(seg) > 4096 else np.pad(seg, (0, 4096-len(seg)))
            processed.append(seg)
        X = np.array(processed)
        diff_X = np.pad(np.diff(X, axis=1), ((0,0),(0,1)))
        self.X = torch.tensor(np.stack([X, diff_X], axis=1), dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx]

# -------------------------- Core Functions --------------------------
def run_m_exe(exe_dir, m_exe, pcm_file):
    original_cwd = os.getcwd()
    os.chdir(exe_dir)
    result = subprocess.run(
        [m_exe, "--pcm", f"../{pcm_file}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=True
    )
    output = result.stdout.decode("gbk", errors="ignore")
    print(output)
    heart_rate = 102.5
    for line in output.split("\n"):
        if "Heart Rate:" in line:
            heart_rate = float(line.split(":")[1].strip().replace("BPM", ""))
    os.chdir(original_cwd)
    return heart_rate

def load_segments(output_dir):
    seg_folder = Path(output_dir)
    seg_files = sorted(seg_folder.glob("segment_*.txt"))
    if not seg_files:
        raise FileNotFoundError(f"Segments not found! Check: {seg_folder}")
    segments = [np.loadtxt(f, dtype=np.float32) for f in seg_files]
    return segments, len(seg_files)

def predict_segments(segments, model_path):
    model = HeartSoundModel().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    loader = DataLoader(HeartSoundDataset(segments), batch_size=4, shuffle=False)
    preds = []
    with torch.no_grad():
        for x in loader:
            preds.extend(torch.argmax(model(x.to(device)), dim=1).cpu().numpy())
    return np.array(preds)

def clean_up(output_dir):
    output_path = Path(output_dir)
    if output_path.exists():
        shutil.rmtree(output_path)
        print("[INFO] Output folder cleaned successfully")

# -------------------------- Main Execution --------------------------
if __name__ == "__main__":
    try:
        # ==============================================
        # ALL PATH CONFIGURATIONS HERE (INSIDE MAIN)
        # ==============================================
        EXE_DIR      = "./model_cnn_gru"    # Folder with m.exe (same level as notebook)
        M_EXE        = "m.exe"              # Signal processing executable
        MODEL_FILE   = "./model_cnn_gru/model.pth"  # Model weights path
        OUTPUT_DIR   = "./model_cnn_gru/output"    # Temp output folder
        PCM_FILE     = "yin5.pcm"           # Target PCM file (same folder as notebook)
        # ==============================================

        print("="*60)
        print("Starting Heart Sound Analysis...")
        print("="*60)
        
        # Run signal processing executable
        heart_rate = run_m_exe(EXE_DIR, M_EXE, PCM_FILE)
        # Load segmented data
        segments, num_segments = load_segments(OUTPUT_DIR)
        print(f"[INFO] Successfully loaded {num_segments} segments")
        # Model prediction
        predictions = predict_segments(segments, MODEL_FILE)
        print(f"[INFO] Prediction finished")
        # Clean temporary files
        clean_up(OUTPUT_DIR)
        
        # Calculate results
        abnormal_num = np.sum(predictions == 1)
        abnormal_percent = abnormal_num / num_segments * 100

        # Print final report
        print("\n" + "="*60)
        print("FINAL ANALYSIS REPORT")
        print("="*60)
        print(f"Detected Heart Rate: {heart_rate} BPM")
        print(f"Total Segments: {num_segments}")
        print(f"Segment Predictions: {predictions}")
        print(f"Abnormal Segments: {abnormal_num} ({abnormal_percent:.1f}%)")

        print("\nDiagnosis Result:")
        if abnormal_percent > 40:
            print("⚠️ WARNING: Abnormal ratio over 40% - POSSIBLE CARDIAC ABNORMALITY DETECTED")
        else:
            print("✅ NORMAL: No obvious cardiac abnormality detected")
        print("="*60)

    except Exception as e:
        print(f"[ERROR] {str(e)}")
        clean_up(OUTPUT_DIR)

Starting Heart Sound Analysis...
Starting processing for: ../yin5.pcm

Processing Complete:
- Original Fs: 11025 Hz
- Target Fs: 1000 Hz
- Duration: 13.0 seconds
- Spikes Detected: 174
- Heart Rate: 117.2 BPM
- Segments Generated: 7
Saved 7 segments to folder: output

[INFO] Successfully loaded 7 segments
[INFO] Prediction finished
[INFO] Output folder cleaned successfully

FINAL ANALYSIS REPORT
Detected Heart Rate: 117.2 BPM
Total Segments: 7
Segment Predictions: [0 0 0 0 0 0 0]
Abnormal Segments: 0 (0.0%)

Diagnosis Result:
✅ NORMAL: No obvious cardiac abnormality detected
